# Query Decomposition RAG
### Splitting a multi-part question into focused sub-questions

Corpus: `OWASP Top 10 for LLM Applications (2025)` — 10 named risk categories (LLM01–LLM10) sharing vocabulary like “risk”, “attack”, “model”, which is exactly what makes naive retrieval struggle.

## Step 1: Build the pipeline

In [1]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

C:\Users\shiva\AppData\Local\Temp\ipykernel_14400\1805325906.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PDF_PATH = "OWASP-Top-10-for-LLMs-v2025.pdf"

pages = PyPDFLoader(PDF_PATH).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

llm = ChatOllama(model="llama3.2:3b", temperature=0)

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

incorrect startxref pointer(1)


parsing for Object Streams


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Loaded 45 pages -> 131 chunks -> 131 vectors


## Step 2: Baseline — single retrieval for a multi-part question
The question below has two distinct information needs. A single embedding biases toward whichever one dominates the query's vector.

In [4]:
query = "How does prompt injection differ from excessive agency, and how do you mitigate each one?"

print("Single-shot retrieval for the full question:")
for doc in vector_store.similarity_search(query, k=4):
    print(f"page {doc.metadata['page']}: {doc.page_content[:120]}...")

Single-shot retrieval for the full question:
page 7: methods of prevention for prompt injection. However, the following measures can mitigate the
impact of prompt injections...
page 6: Types of Prompt Injection Vulnerabilities
Direct Prompt Injections
Direct prompt injections occur when a user's prompt i...
page 6: make LLM outputs more relevant and accurate, research shows that they do not fully mitigate
prompt injection vulnerabili...
page 7: OWASP Top 10 for LLM Applications v2.0
4genai.owasp.org
the model, alters the behavior of the model in unintended or une...


## Step 3: LLM decomposes the question

In [5]:
decompose_prompt = """Break the following question into 2 to 4 focused, self-contained sub-questions,
one for each distinct part. Return exactly one sub-question per line, no numbering.

Question: {query}"""

sub_questions = [q.strip("-* ").strip() for q in llm.invoke(decompose_prompt.format(query=query)).content.strip().split("\n") if q.strip()]

print(f"Decomposed into {len(sub_questions)} sub-questions:")
for q in sub_questions:
    print(f"  - {q}")

Decomposed into 4 sub-questions:
  - What is prompt injection in the context of language models?
  - How does prompt injection affect the behavior of a language model, leading to undesirable outcomes such as generating biased or misleading responses?
  - How can prompt injection be distinguished from excessive agency, which refers to the model's tendency to overstep its intended role and generate responses that are not aligned with human values or ethics?
  - What strategies can be employed to mitigate prompt injection, including techniques for designing more effective prompts, monitoring model behavior, and implementing safeguards to prevent biased or misleading responses?


## Step 4: Retrieve and answer each sub-question independently

In [6]:
sub_answers = []
for q in sub_questions:
    docs = vector_store.similarity_search(q, k=3)
    context = "\n\n".join(doc.page_content for doc in docs)
    answer = llm.invoke(f"Answer based only on the following context:\n\n{context}\n\nQuestion: {q}\nAnswer:").content.strip()
    sub_answers.append((q, answer))
    print(f"\nQ: {q}\nA: {answer}")


Q: What is prompt injection in the context of language models?
A: In the context of language models, prompt injection refers to a vulnerability where a user's input (prompt) directly alters the behavior of the model in unintended or unexpected ways. This can be either intentional (malicious) or unintentional (inadvertent), and it involves manipulating the model's responses through specific inputs to alter its behavior, which can include bypassing safety measures.



Q: How does prompt injection affect the behavior of a language model, leading to undesirable outcomes such as generating biased or misleading responses?
A: Prompt injection affects the behavior of a language model by directly altering its input in unintended or unexpected ways, either intentionally or unintentionally. This can lead to undesirable outcomes such as:

* Generating biased or misleading responses
* Disclosing sensitive information
* Revealing sensitive information about AI system infrastructure or system prompts
* Providing unauthorized access to functions available to the LLM
* Executing arbitrary commands in connected systems
* Manipulating critical decision-making processes

When a prompt is injected into a language model, it can alter its behavior in unintended ways, leading to these undesirable outcomes. This can happen when a user's input directly modifies the model's behavior or when an external source of content is accepted by the model and interpreted in unexpecte


Q: How can prompt injection be distinguished from excessive agency, which refers to the model's tendency to overstep its intended role and generate responses that are not aligned with human values or ethics?
A: Prompt injection cannot be distinguished from excessive agency based on the provided context. The text does not provide a clear definition of how to differentiate between the two concepts.

However, it can be inferred that prompt injection is related to the model's behavior being altered by external input, whereas excessive agency refers to the model generating responses that are not aligned with human values or ethics. But without further information, it is unclear how these two concepts can be distinguished from one another.



Q: What strategies can be employed to mitigate prompt injection, including techniques for designing more effective prompts, monitoring model behavior, and implementing safeguards to prevent biased or misleading responses?
A: To mitigate prompt injection vulnerabilities, several strategies can be employed:

1. **Designing Effective Prompts**: Provide specific instructions about the model's role, capabilities, and limitations within the system prompt. Enforce strict context adherence, limit responses to specific tasks or topics, and instruct the model to ignore attempts to modify core instructions.
2. **Defining and Validating Expected Output Formats**: Specify clear output formats, request detailed reasoning and source citations, and use deterministic code to validate adherence to these formats.
3. **Implementing Input and Output Filtering**: Define sensitive categories and construct rules for identifying and handling such content. Apply semantic filters and use string-checking to scan

## Step 5: Synthesize the sub-answers into one coherent response

In [7]:
synthesis_prompt = f"Combine the following sub-answers into one coherent, complete answer to the original question.\n\nOriginal question: {query}\n\n"
for q, a in sub_answers:
    synthesis_prompt += f"Sub-question: {q}\nSub-answer: {a}\n\n"

print(llm.invoke(synthesis_prompt).content)

Prompt injection refers to a vulnerability in language models where a user's input (prompt) directly alters the behavior of the model in unintended or unexpected ways. This can be either intentional (malicious) or unintentional, and it involves manipulating the model's responses through specific inputs to alter its behavior, which can include bypassing safety measures.

Prompt injection affects the behavior of a language model by directly altering its input in unintended or unexpected ways, leading to undesirable outcomes such as generating biased or misleading responses, disclosing sensitive information, revealing sensitive information about AI system infrastructure or system prompts, providing unauthorized access to functions available to the LLM, executing arbitrary commands in connected systems, and manipulating critical decision-making processes.

Prompt injection cannot be distinguished from excessive agency based on the provided context. The text does not provide a clear definit

## Try it yourself
1. Ask a 3-part question spanning 3 categories and check the sub-question split.
2. Compare the baseline's single answer against the synthesized one for completeness.
3. Try `Recursive` decomposition: feed one sub-question back through the decomposer.